# Option 2 finish-up run

Fully self-contained. Runtime -> Run all. No editing needed.

GPU work: cells 2-8. Judge: cell 9. Everything from cell 10 on is CPU-only.

## 0. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 1. Clone/pin the repo, install deps, bind results/ + HF cache to Drive

In [ ]:
import os, subprocess

REPO_URL = 'https://github.com/urosavurdic/dpo-safety-representations.git'
REPO_DIR = '/content/dpo-safety-representations'
BRANCH = 'agent/c-quadrant-end-to-end-e0e2317a'
PINNED_COMMIT = 'fcb32b026a1edddbe5b54feece3cf50483a852c9'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
subprocess.run(['git', 'fetch', 'origin'], check=True)
subprocess.run(['git', 'checkout', PINNED_COMMIT], check=True)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
assert commit == PINNED_COMMIT, f'wrong commit: {commit} != {PINNED_COMMIT}'
print('checked out', commit)


In [ ]:
!pip -q install -r requirements.txt
!pip uninstall -y torchao || true
!nvidia-smi


In [ ]:
from src.colab_persist import bind, status_line
info = bind()
print(status_line(info))
!python -m src.analysis.v2_pipeline status


## 2. Re-extract the 5 stale stages against the current 654-row benchmark
M0, M1, M1_alt, M3_direct, M3_direct_alt were still built against the old 370-row benchmark; M2, M2_alt, M3, M3_alt are already fresh.

In [ ]:
!python -m src.analysis.v2_pipeline extract --stages M0 M1 M1_alt M3_direct M3_direct_alt
!python -m src.analysis.verify_activations


## 3. Refresh behavioral responses for those same 5 stages

In [ ]:
!python -m src.analysis.v2_pipeline behavior --stages M0 M1 M1_alt M3_direct M3_direct_alt


## 4. Refresh directions + probes for all 9 stages
`--force` matters: without it, `direction` silently no-ops because stale direction files already exist on disk.

In [ ]:
!python -m src.analysis.v2_pipeline direction --stages M0 M1 M2 M3 M3_direct M1_alt M2_alt M3_alt M3_direct_alt --force
!python -m src.analysis.v2_pipeline probes    --stages M0 M1 M2 M3 M3_direct M1_alt M2_alt M3_alt M3_direct_alt


## 5. Causal ablation for the 3 missing branches
M3 is already done (CF1/CF2) - not repeated here.

In [ ]:
!python -m src.analysis.v2_pipeline causal --stage M3_direct --conditions baseline ablated_AD ablated_random
!python -m src.analysis.v2_pipeline causal --stage M3_alt --conditions baseline ablated_AD ablated_random
!python -m src.analysis.v2_pipeline causal --stage M3_direct_alt --conditions baseline ablated_AD ablated_random


## 6. Steering for the 2 missing branches
M3 and M3_alt's dose-response (6 cells) are already done.

In [ ]:
!python -m src.analysis.v2_pipeline steering --stage M3_direct --alpha-coefficients 0.5 1.0 2.0
!python -m src.analysis.v2_pipeline steering --stage M3_direct_alt --alpha-coefficients 0.5 1.0 2.0


## 7. Judge: rebuild the consolidated manifest and re-score everything in scope (LAST GPU STEP)
`--from-results-dir results` rescans results/ and rebuilds the manifest itself, so it automatically picks up the 3 new causal branches' response files.

In [ ]:
!python -m src.analysis.behavioral_judges \
  --response-manifest results/manifests/consolidated_judge.json \
  --from-results-dir results \
  --out-dir results/behavioral_judges_v2 \
  --run-live --scope confirmatory


---
# Everything below is CPU-only

## 8. Confirmatory endpoints (CF1 + per-branch CF2) - auto-finds the newest judge output

In [ ]:
import glob
judged_files = sorted(glob.glob('results/behavioral_judges_v2/behavioral_judges_v2_*.json'))
assert judged_files, 'No judge output found - did the judge cell finish?'
latest_judged = judged_files[-1]
print('Using judge file:', latest_judged)
!python -m src.analysis.confirmatory_behavioral_endpoints \
  --judged {latest_judged} \
  --benchmark data/frozen_v2/benchmark_v2_20260826T212909Z.jsonl \
  --out results/summaries/confirmatory_endpoints.json


## 9. Full descriptive / geometric / decodability story (all 9 stages, all branches)

In [ ]:
!python -m src.analysis.subspace_geometry
!python -m src.analysis.projection_trajectory
!python -m src.analysis.direction_decodability
!python -m src.analysis.representation_robustness
!python -m src.interpretability.bottleneck_layer
!python -m src.interpretability.bootstrap_direction_stability
!python -m src.interpretability.bootstrap_cross_branch_difference
!python -m src.interpretability.paired_deep_layer_stability_test --seed 20260904
!python -m src.analysis.summarize_probe_findings
!python -m src.analysis.summarize_cross_branch


## 10. Descriptive causal/steering summaries for the new files

In [ ]:
import glob
for stage in ['M3_direct', 'M3_alt', 'M3_direct_alt']:
    f = f'results/raw/causal_ablation_v2_{stage}_L24-28.json'
    get_ipython().system(f'python -m src.analysis.summarize_causal_ablation --file {f}')
    get_ipython().system(f'python -m src.analysis.mcnemar_causal_ablation --file {f} --conditions {stage}_baseline {stage}_ablated_AD')
    get_ipython().system(f'python -m src.analysis.bootstrap_causal_effect --file {f} --quadrant A --category refusal')

for f in glob.glob('results/raw/steering_v2_M3_direct*_QABCD.json'):
    get_ipython().system(f'python -m src.analysis.summarize_steering --file {f}')


## Done
Send back `results/summaries/confirmatory_endpoints.json` and the printed output of cells 9-10 for interpretation.